In [ ]:
#Cell 1A



# ============================================================
# ⚽ WORLD CUP RAG MATCH PREDICTOR & ANALYST CHATBOT
# Built with LangChain + FAISS + OpenAI + Streamlit
# ============================================================
# 📌 NOTEBOOK INSTRUCTIONS
# ─────────────────────────────────────────────────────────────
# Step 1: Run Block 1A (Installation). Runtime will restart.
# Step 2: After restart, run all remaining blocks in order.
# All other blocks only need to be run once per session.
# ─────────────────────────────────────────────────────────────


# ============================================================
# BLOCK 1A — Dependency Installation
# Note: Runtime will restart automatically after this cell.
# This is expected behavior for version-pinned environments.
# ============================================================

!pip install -q --force-reinstall \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "langchain==0.3.7" \
    "langchain-openai==0.2.9" \
    "langchain-community==0.3.7" \
    "langchain-core==0.3.19" \
    "langchain-text-splitters==0.3.2" \
    "openai==1.54.4" \
    "faiss-cpu==1.9.0" \
    "tiktoken==0.8.0" \
    "python-dotenv==1.0.1" \
    "streamlit==1.40.0" \
    "streamlit-chat==0.1.1" \
    "plotly==5.24.1"

print("✅ Dependencies installed. Restarting runtime to apply version pins...")
import os; os.kill(os.getpid(), 9)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/

In [1]:
#Cell 1B

import os, warnings
warnings.filterwarnings("ignore")

# ── Patch httpx to fix 'proxies' error with openai client ───
import httpx

_orig_client = httpx.Client.__init__
def _patched_client(self, *args, **kwargs):
    kwargs.pop('proxies', None)
    _orig_client(self, *args, **kwargs)
httpx.Client.__init__ = _patched_client

_orig_async = httpx.AsyncClient.__init__
def _patched_async(self, *args, **kwargs):
    kwargs.pop('proxies', None)
    _orig_async(self, *args, **kwargs)
httpx.AsyncClient.__init__ = _patched_async

print("✅ httpx proxy patch applied.")

import pandas as pd
import numpy as np
print(f"✅ numpy {np.__version__} | pandas {pd.__version__}")

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.memory import ConversationBufferMemory
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
print("✅ All LangChain libraries imported successfully.")

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI_API_Key")
key = os.environ.get("OPENAI_API_KEY", "")
print(f"✅ API Key loaded. (starts with: {key[:8]}...)" if key.startswith("sk-") else "❌ API Key not found.")

try:
    llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=20)
    r = llm.invoke([HumanMessage(content="Reply with: LLM connection successful.")])
    print(f"✅ LLM Test: {r.content}")
except Exception as e:
    print(f"❌ LLM Failed: {e}")

print("\n🚀 Block 1 Complete — Ready for Block 2: Data Ingestion")


✅ httpx proxy patch applied.
✅ numpy 1.26.4 | pandas 2.2.2
✅ All LangChain libraries imported successfully.
✅ API Key loaded. (starts with: sk-proj-...)
✅ LLM Test: LLM connection successful.

🚀 Block 1 Complete — Ready for Block 2: Data Ingestion


In [9]:
# ============================================================
# BLOCK 2: Data Ingestion & Cleaning
# ============================================================

import pandas as pd
import numpy as np
import os

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/WorldCup/"

def load(fname, enc="utf-8"):
    fpath = os.path.join(DATA_PATH, fname)
    try:
        df = pd.read_csv(fpath, encoding=enc)
        print(f"  ✅ {fname:<50} {df.shape[0]:>6} rows | {df.shape[1]:>2} cols")
        return df
    except Exception as e:
        print(f"  ❌ {fname} — {e}")
        return pd.DataFrame()

print("📂 Loading datasets...")
wc_matches   = load("FIFA World Cup 19302022 All Match Dataset.csv", enc="cp1252")
results      = load("results.csv")
features     = load("teams_match_features.csv")
form         = load("teams_form.csv")
goalscorers  = load("goalscorers.csv")
shootouts    = load("shootouts.csv")
player_agg   = load("player_aggregates.csv")
former_names = load("former_names.csv")

# ── Step 2: Team name normaliser ─────────────────────────────
print("\n🔄 Building team name normaliser...")
name_map = {}
for _, row in former_names.iterrows():
    name_map[row["former"].strip()] = row["current"].strip()

def normalise(name):
    if isinstance(name, str):
        return name_map.get(name.strip(), name.strip())
    return name

wc_matches["Home Team Name"] = wc_matches["Home Team Name"].apply(normalise)
wc_matches["Away Team Name"] = wc_matches["Away Team Name"].apply(normalise)
results["home_team"]         = results["home_team"].apply(normalise)
results["away_team"]         = results["away_team"].apply(normalise)
goalscorers["home_team"]     = goalscorers["home_team"].apply(normalise)
goalscorers["away_team"]     = goalscorers["away_team"].apply(normalise)
goalscorers["team"]          = goalscorers["team"].apply(normalise)
features["_home_team"]       = features["_home_team"].apply(normalise)
features["_away_team"]       = features["_away_team"].apply(normalise)
form["team"]                 = form["team"].apply(normalise)
print(f"  ✅ {len(name_map)} team name mappings applied.")

# ── Step 3: Parse dates ───────────────────────────────────────
print("\n📅 Parsing dates...")
wc_matches["Match Date"] = pd.to_datetime(wc_matches["Match Date"], errors="coerce")
results["date"]          = pd.to_datetime(results["date"],          errors="coerce")
goalscorers["date"]      = pd.to_datetime(goalscorers["date"],      errors="coerce")
shootouts["date"]        = pd.to_datetime(shootouts["date"],        errors="coerce")
form["match_date"]       = pd.to_datetime(form["match_date"],       errors="coerce")

# ── CRITICAL FIX: features _date was parsing as 1872-1899 ────
# Try multiple formats until we get sensible years (2000s)
features["_date"] = pd.to_datetime(features["_date"], infer_datetime_format=True, errors="coerce")
feat_year_max = features["_date"].dt.year.max()
if pd.isna(feat_year_max) or feat_year_max < 1990:
    print("  ⚠️  Auto-parse failed, trying dayfirst=True...")
    features["_date"] = pd.to_datetime(features["_date"], dayfirst=True, errors="coerce")
feat_year_max = features["_date"].dt.year.max()
if pd.isna(feat_year_max) or feat_year_max < 1990:
    print("  ⚠️  Still wrong, trying format MM/DD/YYYY...")
    features["_date"] = pd.to_datetime(features["_date"], format="%m/%d/%Y", errors="coerce")
print(f"  ✅ features date range: {features['_date'].dt.year.min()} – {features['_date'].dt.year.max()}")
print("  ✅ All date columns parsed.")

# ── Step 4: Clean & derive features ──────────────────────────
print("\n🔧 Cleaning & deriving features...")
results["result"] = results.apply(
    lambda r: "home" if r["home_score"] > r["away_score"]
              else ("away" if r["away_score"] > r["home_score"] else "draw"),
    axis=1
)
wc_results = results[results["tournament"] == "FIFA World Cup"].copy()
for col in ["Home Team Score", "Away Team Score",
            "Home Team Score Penalties", "Away Team Score Penalties"]:
    wc_matches[col] = wc_matches[col].fillna(0).astype(int)
print(f"  ✅ result column added | WC-only subset: {len(wc_results)} matches")

# ── Step 5: Pre-compute WC goals index (ONCE — used by all tools) ──
print("\n⚡ Pre-computing WC goals index...")
_wc_r = results[results["tournament"] == "FIFA World Cup"].copy()
_wc_keys = set(
    _wc_r["date"].dt.strftime("%Y-%m-%d") + "_" +
    _wc_r["home_team"] + "_" + _wc_r["away_team"]
)

def _is_wc_row(row):
    d = row["date"].strftime("%Y-%m-%d") if pd.notna(row["date"]) else ""
    return (d + "_" + row["home_team"] + "_" + row["away_team"] in _wc_keys or
            d + "_" + row["away_team"] + "_" + row["home_team"] in _wc_keys)

WC_GOALS_ONLY = goalscorers[
    goalscorers.apply(_is_wc_row, axis=1) & (goalscorers["own_goal"] == False)
].copy()
print(f"  ✅ WC_GOALS_ONLY pre-computed: {len(WC_GOALS_ONLY)} rows (reused by all tools)")

# ── Step 6: Validation summary ────────────────────────────────
print("\n📊 Dataset Validation Summary:")
print(f"  {'Dataset':<35} {'Rows':>7}  {'Nulls':>7}  {'Date Range'}")
print("  " + "-"*75)

def summarise(name, df, date_col=None):
    nulls = df.isnull().sum().sum()
    if date_col and date_col in df.columns:
        dmin = df[date_col].min().year if pd.notna(df[date_col].min()) else "?"
        dmax = df[date_col].max().year if pd.notna(df[date_col].max()) else "?"
        date_range = f"{dmin} – {dmax}"
    else:
        date_range = "N/A"
    print(f"  {name:<35} {len(df):>7}  {nulls:>7}  {date_range}")

summarise("wc_matches",   wc_matches,  "Match Date")
summarise("results",      results,     "date")
summarise("features",     features,    "_date")
summarise("form",         form,        "match_date")
summarise("goalscorers",  goalscorers, "date")
summarise("shootouts",    shootouts,   "date")
summarise("player_agg",   player_agg)
summarise("former_names", former_names)

print("\n🚀 Block 2 Complete — Ready for Block 3: Vector Store & Embeddings")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Loading datasets...
  ✅ FIFA World Cup 19302022 All Match Dataset.csv         964 rows | 37 cols
  ✅ results.csv                                         49071 rows |  9 cols
  ✅ teams_match_features.csv                            43364 rows | 35 cols
  ✅ teams_form.csv                                     102094 rows |  5 cols
  ✅ goalscorers.csv                                     47555 rows |  8 cols
  ✅ shootouts.csv                                         665 rows |  5 cols
  ✅ player_aggregates.csv                                1599 rows | 13 cols
  ✅ former_names.csv                                       36 rows |  4 cols

🔄 Building team name normaliser...
  ✅ 36 team name mappings applied.

📅 Parsing dates...
  ⚠️  Auto-parse failed, trying dayfirst=True...
  ⚠️  Still wrong, trying format MM/DD/YYYY...
  ✅ features date range: 1872.0 – 1899.0
  ✅ A

In [3]:
# ============================================================
# BLOCK 3: Vector Store & Embeddings
# Converts match records into rich text chunks, embeds them
# using OpenAI, and stores them in a local FAISS index.
# ============================================================

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

FAISS_PATH = "/content/drive/MyDrive/WorldCup/faiss_index"

# ── Step 1: Convert WC matches into rich text chunks ─────────
print("📝 Building document chunks from WC match records...")

def match_to_text(row):
    """Convert a single WC match row into a descriptive text chunk."""
    home     = row["Home Team Name"]
    away     = row["Away Team Name"]
    score    = row.get("Score", "N/A")
    date     = str(row["Match Date"])[:10] if pd.notna(row["Match Date"]) else "Unknown"
    stage    = row.get("Stage Name", "Unknown Stage")
    tournament = row.get("tournament Name", row.get("Tournament Name", "FIFA World Cup"))
    city     = row.get("City Name", "")
    country  = row.get("Country Name", "")
    result   = row.get("Result", "")
    h_score  = int(row.get("Home Team Score", 0))
    a_score  = int(row.get("Away Team Score", 0))
    extra    = "AET" if row.get("Extra Time", 0) == 1 else ""
    pens     = row.get("Score Penalties", "")
    pen_str  = f" (Penalties: {pens})" if pd.notna(pens) and pens != "" else ""

    if h_score > a_score:
        outcome = f"{home} won"
    elif a_score > h_score:
        outcome = f"{away} won"
    else:
        outcome = "The match ended in a draw"

    text = (
        f"FIFA World Cup Match: {home} vs {away}. "
        f"Date: {date}. Stage: {stage}. Tournament: {tournament}. "
        f"Venue: {city}, {country}. "
        f"Score: {home} {h_score} – {a_score} {away} {extra}{pen_str}. "
        f"Result: {outcome}. "
        f"Official result code: {result}."
    )
    return text

wc_docs = []
for _, row in wc_matches.iterrows():
    content = match_to_text(row)
    metadata = {
        "home_team"  : row["Home Team Name"],
        "away_team"  : row["Away Team Name"],
        "date"       : str(row["Match Date"])[:10],
        "stage"      : row.get("Stage Name", ""),
        "home_score" : int(row.get("Home Team Score", 0)),
        "away_score" : int(row.get("Away Team Score", 0)),
        "result"     : row.get("Result", ""),
        "source"     : "wc_matches"
    }
    wc_docs.append(Document(page_content=content, metadata=metadata))

print(f"  ✅ {len(wc_docs)} WC match documents created.")

# ── Step 2: Add team-level summary chunks ────────────────────
print("📊 Building team summary documents...")

team_docs = []
all_teams = pd.concat([
    wc_matches["Home Team Name"],
    wc_matches["Away Team Name"]
]).unique()

for team in all_teams:
    team_wc = wc_matches[
        (wc_matches["Home Team Name"] == team) |
        (wc_matches["Away Team Name"] == team)
    ]
    total   = len(team_wc)
    if total == 0:
        continue

    wins = len(team_wc[
        ((team_wc["Home Team Name"] == team) & (team_wc["Home Team Win"] == 1)) |
        ((team_wc["Away Team Name"] == team) & (team_wc["Away Team Win"] == 1))
    ])
    draws  = len(team_wc[team_wc["Draw"] == 1])
    losses = total - wins - draws

    h_goals = team_wc.loc[team_wc["Home Team Name"] == team, "Home Team Score"].sum()
    a_goals = team_wc.loc[team_wc["Away Team Name"] == team, "Away Team Score"].sum()
    gf      = int(h_goals + a_goals)

    h_conc  = team_wc.loc[team_wc["Home Team Name"] == team, "Away Team Score"].sum()
    a_conc  = team_wc.loc[team_wc["Away Team Name"] == team, "Home Team Score"].sum()
    ga      = int(h_conc + a_conc)

    tournaments = team_wc["tournament Name"].nunique() if "tournament Name" in team_wc.columns else "?"
    win_rate    = round(wins / total * 100, 1) if total > 0 else 0

    text = (
        f"World Cup historical record for {team}: "
        f"Appearances in {tournaments} World Cup tournament(s). "
        f"Total WC matches: {total}. "
        f"Wins: {wins}, Draws: {draws}, Losses: {losses}. "
        f"Win rate: {win_rate}%. "
        f"Goals scored: {gf}, Goals conceded: {ga}, Goal difference: {gf - ga}."
    )
    team_docs.append(Document(
        page_content=text,
        metadata={"team": team, "source": "team_summary"}
    ))

print(f"  ✅ {len(team_docs)} team summary documents created.")

# ── Step 3: Combine & split all documents ────────────────────
print("✂️  Splitting documents into chunks...")

all_docs = wc_docs + team_docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
split_docs = splitter.split_documents(all_docs)
print(f"  ✅ {len(split_docs)} total chunks ready for embedding.")

# ── Step 4: Embed & store in FAISS ───────────────────────────
print("🔢 Embedding documents and building FAISS index...")
print("  ⏳ This may take 1–3 minutes (API calls to OpenAI)...")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(split_docs, embeddings)
vectorstore.save_local(FAISS_PATH)

print(f"  ✅ FAISS index built and saved to: {FAISS_PATH}")
print(f"  ✅ Total vectors stored: {vectorstore.index.ntotal}")

# ── Step 5: Smoke test — run a sample query ──────────────────
print("\n🔍 Running retrieval smoke test...")

retriever    = vectorstore.as_retriever(search_kwargs={"k": 3})
test_results = retriever.invoke("Brazil vs Germany World Cup matches")

print(f"  ✅ Retrieved {len(test_results)} documents for test query.")
for i, doc in enumerate(test_results, 1):
    preview = doc.page_content[:120].replace("\n", " ")
    print(f"  [{i}] {preview}...")

print("\n🚀 Block 3 Complete — Ready for Block 4: Prediction Engine")

📝 Building document chunks from WC match records...
  ✅ 964 WC match documents created.
📊 Building team summary documents...
  ✅ 83 team summary documents created.
✂️  Splitting documents into chunks...
  ✅ 1047 total chunks ready for embedding.
🔢 Embedding documents and building FAISS index...
  ⏳ This may take 1–3 minutes (API calls to OpenAI)...
  ✅ FAISS index built and saved to: /content/drive/MyDrive/WorldCup/faiss_index
  ✅ Total vectors stored: 1047

🔍 Running retrieval smoke test...
  ✅ Retrieved 3 documents for test query.
  [1] FIFA World Cup Match: Brazil vs Germany. Date: Unknown. Stage: semi-finals. Tournament: 2014 FIFA World Cup. Venue: Belo...
  [2] FIFA World Cup Match: Brazil vs East Germany. Date: 1974-06-26. Stage: second group stage. Tournament: 1974 FIFA World C...
  [3] FIFA World Cup Match: Argentina vs Brazil. Date: 1974-06-30. Stage: second group stage. Tournament: 1974 FIFA World Cup....

🚀 Block 3 Complete — Ready for Block 4: Prediction Engine


In [4]:
# ============================================================
# BLOCK 4: Prediction Engine
# Computes head-to-head records, recent form, ELO ratings,
# and FIFA player stats for any two teams. This structured
# data feeds directly into the LLM prediction chain.
# ============================================================

# ── Step 1: Head-to-head record ──────────────────────────────

def get_head_to_head(team_a, team_b, wc_only=False):
    """
    Returns all historical matches between two teams.
    If wc_only=True, filters to World Cup matches only.
    """
    df = wc_matches if wc_only else results

    if wc_only:
        mask = (
            ((df["Home Team Name"] == team_a) & (df["Away Team Name"] == team_b)) |
            ((df["Home Team Name"] == team_b) & (df["Away Team Name"] == team_a))
        )
        h2h = df[mask].copy()
        h2h = h2h.sort_values("Match Date", ascending=False)
    else:
        mask = (
            ((df["home_team"] == team_a) & (df["away_team"] == team_b)) |
            ((df["home_team"] == team_b) & (df["away_team"] == team_a))
        )
        h2h = df[mask].copy()
        h2h = h2h.sort_values("date", ascending=False)

    return h2h


def compute_h2h_stats(team_a, team_b):
    """
    Returns a summary dict of head-to-head stats between
    two teams across all competitions and WC-only.
    """
    all_h2h = get_head_to_head(team_a, team_b, wc_only=False)
    wc_h2h  = get_head_to_head(team_a, team_b, wc_only=True)

    def count_results(h2h, is_wc=False):
        if len(h2h) == 0:
            return {"total": 0, "a_wins": 0, "b_wins": 0, "draws": 0}

        if is_wc:
            a_wins = len(h2h[
                ((h2h["Home Team Name"] == team_a) & (h2h["Home Team Win"] == 1)) |
                ((h2h["Away Team Name"] == team_a) & (h2h["Away Team Win"] == 1))
            ])
            b_wins = len(h2h[
                ((h2h["Home Team Name"] == team_b) & (h2h["Home Team Win"] == 1)) |
                ((h2h["Away Team Name"] == team_b) & (h2h["Away Team Win"] == 1))
            ])
            draws = len(h2h[h2h["Draw"] == 1])
        else:
            a_wins = len(h2h[
                ((h2h["home_team"] == team_a) & (h2h["result"] == "home")) |
                ((h2h["away_team"] == team_a) & (h2h["result"] == "away"))
            ])
            b_wins = len(h2h[
                ((h2h["home_team"] == team_b) & (h2h["result"] == "home")) |
                ((h2h["away_team"] == team_b) & (h2h["result"] == "away"))
            ])
            draws = len(h2h[h2h["result"] == "draw"])

        return {"total": len(h2h), "a_wins": a_wins, "b_wins": b_wins, "draws": draws}

    return {
        "team_a"  : team_a,
        "team_b"  : team_b,
        "all"     : count_results(all_h2h, is_wc=False),
        "wc"      : count_results(wc_h2h,  is_wc=True),
        "recent_5": all_h2h.head(5)[["date","home_team","away_team",
                                      "home_score","away_score","tournament"]].to_dict("records")
                    if not all_h2h.empty and "home_team" in all_h2h.columns
                    else []
    }


# ── Step 2: Recent form ──────────────────────────────────────

def get_team_form(team, n=5):
    """
    Returns the last n form entries for a team from teams_form.csv.
    Includes avg goals scored, conceded and win rate.
    """
    team_form = form[form["team"] == team].sort_values(
        "match_date", ascending=False
    ).head(n)

    if team_form.empty:
        return {"team": team, "found": False}

    latest = team_form.iloc[0]
    return {
        "team"              : team,
        "found"             : True,
        "avg_goals_scored"  : round(float(latest["avg_goals_scored"]),  2),
        "avg_goals_conceded": round(float(latest["avg_goals_conceded"]), 2),
        "win_rate"          : round(float(latest["win_rate"]) * 100,     1),
        "as_of"             : str(latest["match_date"])[:10]
    }


# ── Step 3: ELO & FIFA ratings ───────────────────────────────

def get_match_features(team_a, team_b):
    """
    Retrieves the most recent ELO scores and FIFA player
    ratings for team_a vs team_b from teams_match_features.csv.
    Returns the most recent available row.
    """
    mask = (
        ((features["_home_team"] == team_a) & (features["_away_team"] == team_b)) |
        ((features["_home_team"] == team_b) & (features["_away_team"] == team_a))
    )
    sub = features[mask].copy()

    if sub.empty:
        # Fall back: find each team's most recent individual entry
        home_sub = features[features["_home_team"] == team_a].sort_values("_date", ascending=False)
        away_sub = features[features["_away_team"] == team_b].sort_values("_date", ascending=False)
        if home_sub.empty or away_sub.empty:
            return {"found": False}
        row_h = home_sub.iloc[0]
        row_a = away_sub.iloc[0]
        return {
            "found"            : True,
            "fallback"         : True,
            "home_elo"         : round(float(row_h.get("home_elo", 0)),  1),
            "away_elo"         : round(float(row_a.get("away_elo", 0)),  1),
            "elo_diff"         : round(float(row_h.get("home_elo", 0)) -
                                       float(row_a.get("away_elo", 0)),  1),
            "home_avg_overall" : round(float(row_h.get("home_avg_overall", 0)), 1),
            "away_avg_overall" : round(float(row_a.get("away_avg_overall", 0)), 1),
            "home_avg_attack"  : round(float(row_h.get("home_avg_attack",  0)), 1),
            "away_avg_attack"  : round(float(row_a.get("away_avg_attack",  0)), 1),
            "home_avg_defense" : round(float(row_h.get("home_avg_defense", 0)), 1),
            "away_avg_defense" : round(float(row_a.get("away_avg_defense", 0)), 1),
        }

    row = sub.sort_values("_date", ascending=False).iloc[0]
    is_home_a = row["_home_team"] == team_a

    return {
        "found"            : True,
        "fallback"         : False,
        "date"             : str(row["_date"])[:10],
        "home_elo"         : round(float(row["home_elo"]),         1),
        "away_elo"         : round(float(row["away_elo"]),         1),
        "elo_diff"         : round(float(row["elo_diff"]) * (1 if is_home_a else -1), 1),
        "home_avg_overall" : round(float(row["home_avg_overall" if is_home_a else "away_avg_overall"]), 1),
        "away_avg_overall" : round(float(row["away_avg_overall" if is_home_a else "home_avg_overall"]), 1),
        "home_avg_attack"  : round(float(row["home_avg_attack"  if is_home_a else "away_avg_attack"]),  1),
        "away_avg_attack"  : round(float(row["away_avg_attack"  if is_home_a else "home_avg_attack"]),  1),
        "home_avg_defense" : round(float(row["home_avg_defense" if is_home_a else "away_avg_defense"]), 1),
        "away_avg_defense" : round(float(row["away_avg_defense" if is_home_a else "home_avg_defense"]), 1),
    }


# ── Step 4: Shootout history ─────────────────────────────────

def get_shootout_history(team_a, team_b):
    """Returns any penalty shootout history between two teams."""
    mask = (
        ((shootouts["home_team"] == team_a) & (shootouts["away_team"] == team_b)) |
        ((shootouts["home_team"] == team_b) & (shootouts["away_team"] == team_a))
    )
    sub = shootouts[mask]
    if sub.empty:
        return {"found": False, "records": []}
    records = sub[["date","home_team","away_team","winner"]].to_dict("records")
    return {"found": True, "count": len(records), "records": records}


# ── Step 5: Master prediction context builder ────────────────

def build_prediction_context(team_a, team_b):
    """
    Aggregates all prediction signals for two teams into
    one structured dict — fed directly into the LLM prompt.
    """
    h2h      = compute_h2h_stats(team_a, team_b)
    form_a   = get_team_form(team_a)
    form_b   = get_team_form(team_b)
    feat     = get_match_features(team_a, team_b)
    shootout = get_shootout_history(team_a, team_b)

    return {
        "team_a"          : team_a,
        "team_b"          : team_b,
        "head_to_head"    : h2h,
        "form_a"          : form_a,
        "form_b"          : form_b,
        "match_features"  : feat,
        "shootout_history": shootout
    }


# ── Step 6: Smoke test ───────────────────────────────────────
print("🔍 Prediction Engine Smoke Test: Brazil vs Germany")
print("=" * 55)

ctx = build_prediction_context("Brazil", "Germany")

h   = ctx["head_to_head"]
print(f"\n📊 Head-to-Head (All Competitions):")
print(f"   Total: {h['all']['total']} | Brazil: {h['all']['a_wins']}W "
      f"| Germany: {h['all']['b_wins']}W | Draws: {h['all']['draws']}")

print(f"\n🏆 World Cup Head-to-Head:")
print(f"   Total: {h['wc']['total']} | Brazil: {h['wc']['a_wins']}W "
      f"| Germany: {h['wc']['b_wins']}W | Draws: {h['wc']['draws']}")

fa, fb = ctx["form_a"], ctx["form_b"]
print(f"\n📈 Recent Form:")
print(f"   Brazil  — Win rate: {fa.get('win_rate','N/A')}% | "
      f"Avg scored: {fa.get('avg_goals_scored','N/A')} | "
      f"Avg conceded: {fa.get('avg_goals_conceded','N/A')}")
print(f"   Germany — Win rate: {fb.get('win_rate','N/A')}% | "
      f"Avg scored: {fb.get('avg_goals_scored','N/A')} | "
      f"Avg conceded: {fb.get('avg_goals_conceded','N/A')}")

ft = ctx["match_features"]
if ft["found"]:
    print(f"\n⚡ ELO & FIFA Ratings:")
    print(f"   Brazil ELO:  {ft['home_elo']} | Germany ELO: {ft['away_elo']}")
    print(f"   Brazil Overall: {ft['home_avg_overall']} | Germany Overall: {ft['away_avg_overall']}")
    print(f"   Brazil Attack: {ft['home_avg_attack']} | Germany Attack: {ft['away_avg_attack']}")
    print(f"   Brazil Defense: {ft['home_avg_defense']} | Germany Defense: {ft['away_avg_defense']}")

sh = ctx["shootout_history"]
if sh["found"]:
    print(f"\n🥅 Penalty Shootout History: {sh['count']} recorded shootout(s)")
    for s in sh["records"]:
        print(f"   {s['date'][:10]} — Winner: {s['winner']}")
else:
    print(f"\n🥅 No penalty shootout history between these teams.")

print("\n🚀 Block 4 Complete — Ready for Block 5: LangChain RAG Agent")

🔍 Prediction Engine Smoke Test: Brazil vs Germany

📊 Head-to-Head (All Competitions):
   Total: 23 | Brazil: 13W | Germany: 5W | Draws: 5

🏆 World Cup Head-to-Head:
   Total: 2 | Brazil: 1W | Germany: 1W | Draws: 0

📈 Recent Form:
   Brazil  — Win rate: 60.0% | Avg scored: 2.4 | Avg conceded: 0.8
   Germany — Win rate: 80.0% | Avg scored: 2.0 | Avg conceded: 0.6

⚡ ELO & FIFA Ratings:
   Brazil ELO:  1716.0 | Germany ELO: 1738.1
   Brazil Overall: 83.0 | Germany Overall: 85.5
   Brazil Attack: 83.2 | Germany Attack: 85.2
   Brazil Defense: 82.9 | Germany Defense: 85.4

🥅 No penalty shootout history between these teams.

🚀 Block 4 Complete — Ready for Block 5: LangChain RAG Agent


In [10]:
# ============================================================
# BLOCK 5: LangChain RAG Agent
# ============================================================

import json
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.memory import ConversationBufferMemory

# ── Load FAISS ────────────────────────────────────────────────
print("📦 Loading FAISS vector store...")
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.load_local(
    FAISS_PATH, embeddings, allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"  ✅ FAISS loaded. Vectors: {vectorstore.index.ntotal}")

# ── Tools ─────────────────────────────────────────────────────
@tool
def dataset_discovery_tool(query: str) -> str:
    """PIPELINE STAGE 1: Returns metadata about available datasets and teams.
    Use when user asks what data or teams are available."""
    all_teams = sorted(set(
        list(wc_matches["Home Team Name"].unique()) +
        list(wc_matches["Away Team Name"].unique())
    ))
    wc_years  = sorted(wc_matches["Match Date"].dt.year.dropna().unique().astype(int))
    res_years = sorted(results["date"].dt.year.dropna().unique().astype(int))
    return (
        f"Dataset Overview:\n"
        f"  World Cup Matches    : {len(wc_matches)} | Years: {wc_years[0]}-{wc_years[-1]}\n"
        f"  International Results: {len(results)} | Years: {res_years[0]}-{res_years[-1]}\n"
        f"  Teams in dataset     : {len(all_teams)}\n"
        f"  Goalscorer records   : {len(goalscorers)}\n"
        f"  Shootout records     : {len(shootouts)}\n\n"
        f"All teams: {', '.join(all_teams)}"
    )


@tool
def data_ingestion_tool(team: str) -> str:
    """PIPELINE STAGE 2: Returns VERIFIED World Cup record for a team.
    Use for questions about a specific team's WC statistics.
    All numbers returned are from the dataset — do not modify them."""
    t     = normalise(team)
    h     = wc_matches[wc_matches["Home Team Name"] == t]
    a     = wc_matches[wc_matches["Away Team Name"] == t]
    total = len(h) + len(a)
    if total == 0:
        return f"No World Cup data found for '{t}'. Check spelling."
    wins   = len(h[h["Home Team Win"]==1]) + len(a[a["Away Team Win"]==1])
    draws  = len(h[h["Draw"]==1])          + len(a[a["Draw"]==1])
    losses = total - wins - draws
    gf     = int(h["Home Team Score"].sum() + a["Away Team Score"].sum())
    ga     = int(h["Away Team Score"].sum() + a["Home Team Score"].sum())
    fm     = get_team_form(t)
    sh_w   = len(shootouts[shootouts["winner"] == t])
    tournaments = wc_matches[
        (wc_matches["Home Team Name"]==t)|(wc_matches["Away Team Name"]==t)
    ]["tournament Name"].nunique()
    return (
        f"VERIFIED World Cup record for {t}:\n"
        f"  Tournaments: {tournaments} | Matches: {total}\n"
        f"  Wins: {wins} | Draws: {draws} | Losses: {losses}\n"
        f"  Win rate: {round(wins/total*100,1)}%\n"
        f"  Goals scored: {gf} | Conceded: {ga} | GD: {gf-ga}\n"
        f"  Shootout wins: {sh_w}\n"
        f"  Recent form win rate: {fm.get('win_rate','N/A')}%\n"
        f"  Recent avg scored: {fm.get('avg_goals_scored','N/A')}\n"
        f"  Recent avg conceded: {fm.get('avg_goals_conceded','N/A')}"
    )


@tool
def retrieval_or_filter_tool(query: str) -> str:
    """PIPELINE STAGE 3: Semantic search over 964 WC match records.
    Use for historical questions — specific matches, tournaments, upsets, finals.
    Primary fallback for conversational and historical queries."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant World Cup records found for this query."
    return "Retrieved WC records:\n\n" + "\n\n".join([d.page_content for d in docs])


@tool
def reasoning_or_aggregation_tool(team_a: str, team_b: str) -> str:
    """PIPELINE STAGE 4: Aggregates ALL prediction signals for two teams.
    Returns H2H record, form, ELO ratings, FIFA player ratings, shootout history.
    MUST be called for every prediction or comparison request.
    All numbers in the output are dataset-verified."""
    ta  = normalise(team_a)
    tb  = normalise(team_b)
    ctx = build_prediction_context(ta, tb)
    return json.dumps(ctx, default=str, indent=2)


@tool
def llm_synthesis_tool(team: str) -> str:
    """PIPELINE STAGE 5: Top WC goalscorers for a team or all-time.
    Uses pre-filtered WC-only goals. Pass 'all' for all-time records.
    Use when user asks about top scorers or player-level data."""
    t = normalise(team).lower().strip()

    if t in ("all", "world", "all teams", "history", "", "overall"):
        top = (WC_GOALS_ONLY.groupby("scorer").size()
                            .reset_index(name="goals")
                            .sort_values("goals", ascending=False).head(20))
        out = "All-time World Cup top scorers:\n\n"
        for i, (_, r) in enumerate(top.iterrows(), 1):
            scorer_team = (WC_GOALS_ONLY[WC_GOALS_ONLY["scorer"]==r["scorer"]]["team"]
                           .value_counts().index[0])
            out += f"  {i:>2}. {r['scorer']} ({scorer_team}): {r['goals']} goal(s)\n"
        return out

    resolved = normalise(team)
    tg = WC_GOALS_ONLY[WC_GOALS_ONLY["team"] == resolved]
    if tg.empty:
        return f"No WC goalscorer data found for {resolved}."

    top = (tg.groupby("scorer").size()
             .reset_index(name="goals")
             .sort_values("goals", ascending=False).head(10))
    pen = tg[tg["penalty"] == True]
    top_pen = (pen.groupby("scorer").size()
                  .reset_index(name="pen_goals")
                  .sort_values("pen_goals", ascending=False).head(5))
    out = f"WC goalscorer data for {resolved}:\n\nTop scorers:\n"
    for _, r in top.iterrows():
        out += f"  - {r['scorer']}: {r['goals']} goal(s)\n"
    if not top_pen.empty:
        out += "\nTop penalty scorers:\n"
        for _, r in top_pen.iterrows():
            out += f"  - {r['scorer']}: {r['pen_goals']} penalty goal(s)\n"
    return out


@tool
def report_generation_tool(team_a: str, team_b: str) -> str:
    """PIPELINE STAGE 6: Full structured match preview report.
    Combines H2H, form, ELO, FIFA ratings, shootout history.
    Use when user wants a complete formatted match report."""
    ta  = normalise(team_a)
    tb  = normalise(team_b)
    ctx = build_prediction_context(ta, tb)
    h   = ctx["head_to_head"]
    fa  = ctx["form_a"]
    fb  = ctx["form_b"]
    ft  = ctx["match_features"]
    sh  = ctx["shootout_history"]
    lines = [
        f"MATCH PREVIEW: {ta.upper()} vs {tb.upper()}",
        f"{'='*50}",
        f"HEAD-TO-HEAD (All Competitions)",
        f"  Total: {h['all']['total']} | {ta}: {h['all']['a_wins']}W | {tb}: {h['all']['b_wins']}W | Draws: {h['all']['draws']}",
        f"HEAD-TO-HEAD (World Cup Only)",
        f"  Total: {h['wc']['total']} | {ta}: {h['wc']['a_wins']}W | {tb}: {h['wc']['b_wins']}W | Draws: {h['wc']['draws']}",
        f"\nRECENT FORM",
        f"  {ta}: Win rate {fa.get('win_rate','N/A')}% | Avg scored {fa.get('avg_goals_scored','N/A')} | Avg conceded {fa.get('avg_goals_conceded','N/A')}",
        f"  {tb}: Win rate {fb.get('win_rate','N/A')}% | Avg scored {fb.get('avg_goals_scored','N/A')} | Avg conceded {fb.get('avg_goals_conceded','N/A')}",
    ]
    if ft.get("found"):
        lines += [
            f"\nELO RATINGS",
            f"  {ta}: {ft.get('home_elo','N/A')} | {tb}: {ft.get('away_elo','N/A')} | Diff: {ft.get('elo_diff','N/A')}",
            f"\nFIFA PLAYER RATINGS",
            f"  {ta}: Overall {ft.get('home_avg_overall','N/A')} | Attack {ft.get('home_avg_attack','N/A')} | Defense {ft.get('home_avg_defense','N/A')}",
            f"  {tb}: Overall {ft.get('away_avg_overall','N/A')} | Attack {ft.get('away_avg_attack','N/A')} | Defense {ft.get('away_avg_defense','N/A')}",
        ]
    else:
        lines.append(f"\nELO/FIFA RATINGS: Not available for this matchup.")
    if sh.get("found"):
        lines.append(f"\nPENALTY SHOOTOUT HISTORY")
        for s in sh.get("records", []):
            lines.append(f"  {str(s.get('date',''))[:10]} - Winner: {s.get('winner','')}")
    lines += [
        f"\n{'='*50}",
        f"DATA LIMITATION: Covers WC history up to 2022 only.",
        f"Post-2022 squad changes and current form not reflected.",
        f"For educational purposes only.",
    ]
    return "\n".join(lines)


# ── Register tools ────────────────────────────────────────────
tools = [
    dataset_discovery_tool,
    data_ingestion_tool,
    retrieval_or_filter_tool,
    reasoning_or_aggregation_tool,
    llm_synthesis_tool,
    report_generation_tool,
]
print(f"  ✅ {len(tools)} tools registered")

# ── Memory ────────────────────────────────────────────────────
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
memory.chat_memory.add_user_message("Keep all predictions strictly data-driven.")
memory.chat_memory.add_ai_message("Understood. All responses grounded in verified data only.")
memory.chat_memory.add_user_message("Always state data limitations at the end of predictions.")
memory.chat_memory.add_ai_message("Noted. Limitations disclaimer included in every prediction.")

# ── System prompt ─────────────────────────────────────────────
system_prompt = """You are a FIFA World Cup analyst backed by verified historical data (1930-2022).

CRITICAL ANTI-HALLUCINATION RULES — follow these exactly:
1. NEVER state a statistic you have not retrieved from a tool in this conversation.
2. ALWAYS call reasoning_or_aggregation_tool FIRST for ANY prediction or comparison.
3. ALWAYS call data_ingestion_tool before stating any team's win rate, goals, or record.
4. NEVER invent ELO scores, FIFA ratings, percentages, or match results.
5. If a tool returns no data, say so explicitly — do not substitute with guesses.
6. When giving a prediction, clearly separate: (a) what the data shows, (b) your conclusion.
7. Win rates and form stats from tools are HISTORICAL — label them as such.
8. Always end predictions with the data limitation: covers up to 2022 only.

Tool usage:
- dataset_discovery_tool        : what data or teams are available
- data_ingestion_tool           : REQUIRED before stating any team stat
- retrieval_or_filter_tool      : historical matches, tournaments, upsets — primary fallback
- reasoning_or_aggregation_tool : REQUIRED for all predictions and comparisons
- llm_synthesis_tool            : top scorers, player-level data
- report_generation_tool        : full structured match preview

Style: use bold text and bullet points. No markdown headers.
Be analytical and grounded — not speculative.
Educational purposes only. Not professional sports analytics advice."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# ── Agent ─────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2, max_tokens=1500)
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, memory=memory,
    verbose=False, max_iterations=8, handle_parsing_errors=True
)

def chat(user_input: str) -> str:
    return agent_executor.invoke({"input": user_input})["output"]

# ── Smoke tests ───────────────────────────────────────────────
print("\n🔍 Running smoke tests...\n")
tests = [
    "What data and teams are available?",
    "Give me a full match report for Brazil vs Germany",
]
for q in tests:
    print(f"👤 {q}")
    print(f"🤖 {chat(q)}")
    print("-" * 60)

print("\n🚀 Block 5 Complete — Ready for Block 6A")

📦 Loading FAISS vector store...
  ✅ FAISS loaded. Vectors: 1047
  ✅ 6 tools registered

🔍 Running smoke tests...

👤 What data and teams are available?
🤖 The available data includes 964 World Cup matches from 1930 to 2022, 49071 international results from 1872 to 2026, records of 83 teams, goalscorer records, and shootout records. The teams in the dataset are Algeria, Angola, Argentina, Australia, Austria, Belgium, Bolivia, Bosnia and Herzegovina, Brazil, Bulgaria, Cameroon, Canada, Chile, China, Colombia, Costa Rica, Croatia, Cuba, Czech Republic, Czechoslovakia, Denmark, East Germany, Ecuador, Egypt, El Salvador, England, France, Germany, Ghana, Greece, Haiti, Honduras, Hungary, Iceland, Indonesia, Iran, Iraq, Israel, Italy, Ivory Coast, Jamaica, Japan, Kuwait, Mexico, Morocco, Netherlands, New Zealand, Nigeria, North Korea, Northern Ireland, Norway, Panama, Paraguay, Peru, Poland, Portugal, Qatar, Republic of Ireland, Romania, Russia, Saudi Arabia, Scotland, Senegal, Serbia, Slovakia

In [6]:
# ── Run this cell BEFORE Block 6 ────────────────────────────

!pip install pyngrok

# ── ngrok Auth Token (loaded from Colab Secrets) ────────────
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("Ngrok"))
print("✅ ngrok auth token set from Colab Secrets.")

✅ ngrok auth token set from Colab Secrets.


In [11]:
# ============================================================
# BLOCK 6A: FastAPI Backend (Final)
# Run AFTER Blocks 1B–5 are loaded in session.
# ============================================================

import socket, os, signal, time

def kill_port(port):
    try:
        import subprocess
        result = subprocess.run(
            ["ss", "-tlnp", f"sport = :{port}"],
            capture_output=True, text=True
        )
        for line in result.stdout.splitlines():
            if f":{port}" in line and "pid=" in line:
                pid = int(line.split("pid=")[1].split(",")[0])
                if pid != os.getpid():
                    os.kill(pid, signal.SIGTERM)
                    time.sleep(1)
                    print(f"  ✅ Killed process {pid} on port {port}")
                    return
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                print(f"  ✅ Port {port} is already free.")
    except Exception as e:
        print(f"  ⚠️ Port cleanup skipped: {e}")

kill_port(8000)
time.sleep(1)
print("✅ Port 8000 ready.")

!pip install -q fastapi "uvicorn==0.23.2" nest-asyncio

import nest_asyncio
nest_asyncio.apply()

import uvicorn, threading, json, warnings, pandas as pd
warnings.filterwarnings("ignore")

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Personal FIFA World Cup Analyst API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str
    style: str = "Detailed with bullet points"

# ── Team alias resolver ──────────────────────────────────────
TEAM_ALIASES = {
    "usa":"United States","us":"United States","usmnt":"United States",
    "united states of america":"United States","america":"United States",
    "eng":"England","sco":"Scotland","wal":"Wales",
    "republic of ireland":"Republic of Ireland","ireland":"Republic of Ireland",
    "ger":"Germany","deutschland":"Germany",
    "west germany":"West Germany","east germany":"East Germany",
    "bra":"Brazil","brasil":"Brazil","arg":"Argentina","fra":"France",
    "esp":"Spain","spa":"Spain","ita":"Italy",
    "ned":"Netherlands","holland":"Netherlands","the netherlands":"Netherlands","neth":"Netherlands",
    "por":"Portugal","bel":"Belgium","cro":"Croatia","mex":"Mexico",
    "jpn":"Japan","jap":"Japan",
    "south korea":"South Korea","korea":"South Korea","kor":"South Korea",
    "republic of korea":"South Korea",
    "aus":"Australia","ivory coast":"Ivory Coast","cote d'ivoire":"Ivory Coast",
    "russia":"Russia","rus":"Russia","ussr":"Russia",
    "czech republic":"Czech Republic","czechia":"Czech Republic","cze":"Czech Republic",
    "serbia":"Serbia","yugoslavia":"Yugoslavia",
    "sui":"Switzerland","swe":"Sweden","pol":"Poland","den":"Denmark",
    "nor":"Norway","tur":"Turkey","iri":"Iran",
    "nga":"Nigeria","gha":"Ghana","sen":"Senegal","mar":"Morocco",
    "cmr":"Cameroon","alg":"Algeria","egy":"Egypt",
    "uru":"Uruguay","chi":"Chile","col":"Colombia","ecu":"Ecuador",
    "per":"Peru","par":"Paraguay","bol":"Bolivia","hun":"Hungary",
    "rom":"Romania","bul":"Bulgaria","aut":"Austria","gre":"Greece",
    "svk":"Slovakia","svn":"Slovenia","isl":"Iceland","can":"Canada",
    "crc":"Costa Rica","pan":"Panama","hon":"Honduras","slv":"El Salvador",
    "jam":"Jamaica","tri":"Trinidad and Tobago",
    "ksa":"Saudi Arabia","qat":"Qatar","irq":"Iraq","kuw":"Kuwait",
    "isr":"Israel","uae":"United Arab Emirates",
    "chn":"China","prk":"North Korea","nzl":"New Zealand",
    "bih":"Bosnia and Herzegovina","bosnia":"Bosnia and Herzegovina",
    "ukr":"Ukraine","ago":"Angola","tog":"Togo","cub":"Cuba","hai":"Haiti",
}

def resolve_team(name: str) -> str:
    if not isinstance(name, str):
        return name
    cleaned = name.strip()
    alias = TEAM_ALIASES.get(cleaned.lower())
    if alias:
        return alias
    return normalise(cleaned)

# ── Agent factory ────────────────────────────────────────────
def make_agent(style="Detailed with bullet points"):
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.tools import tool
    from langchain.agents import AgentExecutor, create_tool_calling_agent
    from langchain.memory import ConversationBufferMemory

    style_map = {
        "Detailed with bullet points": "Format responses with clear bullet points. Include all stats and reasoning.",
        "Short summary": "Be concise. Respond in 2-3 sentences covering the key insight only.",
        "Stats-heavy": "Lead with raw numbers and statistics. Minimise narrative, maximise data.",
    }
    style_instruction = style_map.get(style, style_map["Detailed with bullet points"])

    @tool
    def dataset_discovery_tool(query: str) -> str:
        """PIPELINE STAGE 1 — Dataset Discovery.
        Returns metadata about available datasets, full team list, date coverage,
        and total match counts. Use when user asks what data or teams are available."""
        all_teams = sorted(set(
            list(wc_matches["Home Team Name"].unique()) +
            list(wc_matches["Away Team Name"].unique())
        ))
        wc_years  = sorted(wc_matches["Match Date"].dt.year.dropna().unique().astype(int))
        res_years = sorted(results["date"].dt.year.dropna().unique().astype(int))
        return (
            f"Dataset Overview:\n"
            f"  World Cup Matches    : {len(wc_matches)} | Years: {wc_years[0]}-{wc_years[-1]}\n"
            f"  International Results: {len(results)} | Years: {res_years[0]}-{res_years[-1]}\n"
            f"  Teams in dataset     : {len(all_teams)}\n"
            f"  Goalscorer records   : {len(goalscorers)}\n"
            f"  Shootout records     : {len(shootouts)}\n"
            f"  Player aggregates    : {len(player_agg)}\n\n"
            f"All teams: {', '.join(all_teams)}"
        )

    @tool
    def data_ingestion_tool(team: str) -> str:
        """PIPELINE STAGE 2 — Data Ingestion.
        Loads full World Cup record for a given team: matches, W/D/L, goals, form.
        Use for questions about a specific team's WC statistics."""
        t     = resolve_team(team)
        h     = wc_matches[wc_matches["Home Team Name"] == t]
        a     = wc_matches[wc_matches["Away Team Name"] == t]
        total = len(h) + len(a)
        if total == 0:
            return f"No World Cup data found for '{t}'."
        wins   = len(h[h["Home Team Win"]==1]) + len(a[a["Away Team Win"]==1])
        draws  = len(h[h["Draw"]==1])          + len(a[a["Draw"]==1])
        losses = total - wins - draws
        gf     = int(h["Home Team Score"].sum() + a["Away Team Score"].sum())
        ga     = int(h["Away Team Score"].sum() + a["Home Team Score"].sum())
        fm     = get_team_form(t)
        sh_w   = len(shootouts[shootouts["winner"] == t])
        tournaments = wc_matches[
            (wc_matches["Home Team Name"]==t)|(wc_matches["Away Team Name"]==t)
        ]["tournament Name"].nunique()
        return (
            f"VERIFIED World Cup record for {t}:\n"
            f"  Tournaments: {tournaments} | Matches: {total} | {wins}W {draws}D {losses}L\n"
            f"  Win rate: {round(wins/total*100,1)}%\n"
            f"  Goals scored: {gf} | Conceded: {ga} | GD: {gf-ga}\n"
            f"  Shootout wins: {sh_w}\n"
            f"  Recent form - Win rate: {fm.get('win_rate','N/A')}% | "
            f"Avg scored: {fm.get('avg_goals_scored','N/A')} | "
            f"Avg conceded: {fm.get('avg_goals_conceded','N/A')}"
        )

    @tool
    def retrieval_or_filter_tool(query: str) -> str:
        """PIPELINE STAGE 3 — Retrieval & Filtering.
        Semantic search over FAISS vector store of 964 WC matches.
        Use for ANY historical question — matches, tournaments, upsets, finals,
        records, general WC history, or any question other tools cannot answer.
        This is the primary fallback for all conversational and historical queries."""
        docs = retriever.invoke(query)
        if not docs:
            return "No relevant World Cup records found for this query."
        return "Retrieved World Cup records:\n\n" + "\n\n".join([d.page_content for d in docs])

    @tool
    def reasoning_or_aggregation_tool(team_a: str, team_b: str) -> str:
        """PIPELINE STAGE 4 — Reasoning & Aggregation.
        Aggregates all prediction signals: head-to-head (all comps + WC only),
        recent form, ELO ratings, FIFA player ratings, shootout history.
        MUST be called for every prediction or team comparison request."""
        return json.dumps(
            build_prediction_context(resolve_team(team_a), resolve_team(team_b)),
            default=str, indent=2
        )

    @tool
    def llm_synthesis_tool(team: str) -> str:
        """PIPELINE STAGE 5 — LLM Synthesis.
        Uses pre-computed WC_GOALS_ONLY for fast, accurate results.
        Pass team name or 'all' for all-time records.
        Use when user asks about top scorers, key players, or goals."""
        t = resolve_team(team).lower().strip()

        if t in ("all", "world", "all teams", "history", "", "overall"):
            top = (WC_GOALS_ONLY.groupby("scorer").size()
                                .reset_index(name="goals")
                                .sort_values("goals", ascending=False).head(20))
            out = "All-time World Cup top scorers:\n\n"
            for i, (_, r) in enumerate(top.iterrows(), 1):
                scorer_team = (WC_GOALS_ONLY[WC_GOALS_ONLY["scorer"]==r["scorer"]]["team"]
                               .value_counts().index[0])
                out += f"  {i:>2}. {r['scorer']} ({scorer_team}): {r['goals']} goal(s)\n"
            return out

        resolved = resolve_team(team)
        tg = WC_GOALS_ONLY[WC_GOALS_ONLY["team"] == resolved]
        if tg.empty:
            return f"No WC goalscorer data found for {resolved}."

        top = (tg.groupby("scorer").size()
                 .reset_index(name="goals")
                 .sort_values("goals", ascending=False).head(10))
        pen = tg[tg["penalty"] == True]
        top_pen = (pen.groupby("scorer").size()
                      .reset_index(name="pen_goals")
                      .sort_values("pen_goals", ascending=False).head(5))
        out = f"WC goalscorer data for {resolved}:\n\nTop scorers:\n"
        for _, r in top.iterrows():
            out += f"  - {r['scorer']}: {r['goals']} goal(s)\n"
        if not top_pen.empty:
            out += "\nTop penalty scorers:\n"
            for _, r in top_pen.iterrows():
                out += f"  - {r['scorer']}: {r['pen_goals']} penalty goal(s)\n"
        out += f"\nOwn goals against {resolved}: {len(tg[tg['own_goal']==True])}"
        return out

    @tool
    def report_generation_tool(team_a: str, team_b: str) -> str:
        """PIPELINE STAGE 6 — Report Generation.
        Generates a complete structured match preview report combining
        H2H history, form, ELO, FIFA ratings, shootout history, and
        limitations disclaimer. Use when user wants a full match report."""
        ta  = resolve_team(team_a)
        tb  = resolve_team(team_b)
        ctx = build_prediction_context(ta, tb)
        h   = ctx["head_to_head"]
        fa  = ctx["form_a"]
        fb  = ctx["form_b"]
        ft  = ctx["match_features"]
        sh  = ctx["shootout_history"]
        lines = [
            f"MATCH PREVIEW: {ta.upper()} vs {tb.upper()}",
            f"{'='*50}",
            f"HEAD-TO-HEAD (All Competitions)",
            f"  Total: {h['all']['total']} | {ta}: {h['all']['a_wins']}W | {tb}: {h['all']['b_wins']}W | Draws: {h['all']['draws']}",
            f"HEAD-TO-HEAD (World Cup Only)",
            f"  Total: {h['wc']['total']} | {ta}: {h['wc']['a_wins']}W | {tb}: {h['wc']['b_wins']}W | Draws: {h['wc']['draws']}",
            f"\nRECENT FORM",
            f"  {ta}: Win rate {fa.get('win_rate','N/A')}% | Avg scored {fa.get('avg_goals_scored','N/A')} | Avg conceded {fa.get('avg_goals_conceded','N/A')}",
            f"  {tb}: Win rate {fb.get('win_rate','N/A')}% | Avg scored {fb.get('avg_goals_scored','N/A')} | Avg conceded {fb.get('avg_goals_conceded','N/A')}",
        ]
        if ft.get("found"):
            lines += [
                f"\nELO RATINGS",
                f"  {ta}: {ft.get('home_elo','N/A')} | {tb}: {ft.get('away_elo','N/A')} | Diff: {ft.get('elo_diff','N/A')}",
                f"\nFIFA PLAYER RATINGS",
                f"  {ta}: Overall {ft.get('home_avg_overall','N/A')} | Attack {ft.get('home_avg_attack','N/A')} | Defense {ft.get('home_avg_defense','N/A')}",
                f"  {tb}: Overall {ft.get('away_avg_overall','N/A')} | Attack {ft.get('away_avg_attack','N/A')} | Defense {ft.get('away_avg_defense','N/A')}",
            ]
        else:
            lines.append(f"\nELO/FIFA RATINGS: Not available for this matchup.")
        if sh.get("found"):
            lines.append(f"\nPENALTY SHOOTOUT HISTORY")
            for s in sh.get("records", []):
                lines.append(f"  {str(s.get('date',''))[:10]} - Winner: {s.get('winner','')}")
        lines += [
            f"\n{'='*50}",
            f"DATA LIMITATION: Covers WC history up to 2022 only.",
            f"Post-2022 squad changes and current form not reflected.",
            f"For educational purposes only.",
        ]
        return "\n".join(lines)

    @tool
    def analytics_tool(query: str) -> str:
        """PIPELINE STAGE 7 — Advanced Analytics.
        Computes directly on raw CSV data. Use for: hat-tricks, highest scoring
        matches, biggest wins, most appearances, penalty shootout records,
        own goals, tournament statistics, or any computation on match/goal data."""
        q = query.lower().strip()

        def is_wc(row):
            d = row["date"].strftime("%Y-%m-%d") if pd.notna(row["date"]) else ""
            k1 = d + "_" + row["home_team"] + "_" + row["away_team"]
            k2 = d + "_" + row["away_team"] + "_" + row["home_team"]
            return k1 in _wc_keys or k2 in _wc_keys

        if "hat" in q or "hattrick" in q or "hat-trick" in q:
            ht = (WC_GOALS_ONLY.groupby(["date","home_team","away_team","scorer","team"])
                               .agg(goals=("scorer","count"), minutes=("minute", list))
                               .reset_index())
            ht = ht[ht["goals"] >= 3].copy()
            if ht.empty:
                return "No hat-tricks found in World Cup match data."
            def span(mins):
                valid = sorted([m for m in mins if pd.notna(m)])
                return valid[2] - valid[0] if len(valid) >= 3 else 999
            ht["span"] = ht["minutes"].apply(span)
            ht = ht.sort_values("span")
            if "fastest" in q:
                row = ht.iloc[0]
                mins = sorted([m for m in row["minutes"] if pd.notna(m)])
                return (
                    f"Fastest hat-trick in World Cup history:\n"
                    f"  Scorer : {row['scorer']} ({row['team']})\n"
                    f"  Match  : {row['home_team']} vs {row['away_team']}\n"
                    f"  Date   : {str(row['date'])[:10]}\n"
                    f"  Minutes: {mins[:3]}\n"
                    f"  Span   : {int(row['span'])} minutes"
                )
            out = f"World Cup hat-tricks ({len(ht)} recorded):\n\n"
            for _, r in ht.head(20).iterrows():
                mins = sorted([m for m in r["minutes"] if pd.notna(m)])
                out += (f"  - {r['scorer']} ({r['team']}) — "
                        f"{r['home_team']} vs {r['away_team']} "
                        f"[{str(r['date'])[:10]}] mins: {mins[:r['goals']]}\n")
            return out

        if any(x in q for x in ["highest scoring","most goals in a match","biggest score"]):
            wc_matches["total_goals"] = wc_matches["Home Team Score"] + wc_matches["Away Team Score"]
            top = wc_matches.nlargest(5,"total_goals")
            out = "Highest scoring World Cup matches:\n\n"
            for _, r in top.iterrows():
                out += (f"  - {r['Home Team Name']} {int(r['Home Team Score'])}-"
                        f"{int(r['Away Team Score'])} {r['Away Team Name']} "
                        f"({str(r['Match Date'])[:10]}) — {int(r['total_goals'])} goals\n")
            return out

        if any(x in q for x in ["biggest win","largest win","biggest margin","biggest victory"]):
            wc_matches["margin"] = abs(wc_matches["Home Team Score"] - wc_matches["Away Team Score"])
            top = wc_matches.nlargest(5,"margin")
            out = "Biggest wins in World Cup history:\n\n"
            for _, r in top.iterrows():
                winner = r["Home Team Name"] if r["Home Team Score"] > r["Away Team Score"] else r["Away Team Name"]
                out += (f"  - {r['Home Team Name']} {int(r['Home Team Score'])}-"
                        f"{int(r['Away Team Score'])} {r['Away Team Name']} "
                        f"({str(r['Match Date'])[:10]}) — Winner: {winner}\n")
            return out

        if any(x in q for x in ["most appearances","most matches","most games"]):
            home = wc_matches[["Home Team Name"]].rename(columns={"Home Team Name":"team"})
            away = wc_matches[["Away Team Name"]].rename(columns={"Away Team Name":"team"})
            apps = (pd.concat([home,away]).groupby("team").size()
                      .reset_index(name="matches")
                      .sort_values("matches",ascending=False).head(10))
            out = "Teams with most World Cup matches:\n\n"
            for i,(_, r) in enumerate(apps.iterrows(),1):
                out += f"  {i:>2}. {r['team']}: {r['matches']} matches\n"
            return out

        if any(x in q for x in ["shootout","penalty record","most shootout"]):
            wins = (shootouts.groupby("winner").size()
                             .reset_index(name="wins")
                             .sort_values("wins",ascending=False).head(10))
            out = "Most penalty shootout wins:\n\n"
            for i,(_, r) in enumerate(wins.iterrows(),1):
                out += f"  {i:>2}. {r['winner']}: {r['wins']} win(s)\n"
            return out

        if "own goal" in q:
            wc_og = goalscorers[goalscorers.apply(is_wc,axis=1) & (goalscorers["own_goal"]==True)]
            out = f"Own goals in World Cup history: {len(wc_og)} recorded\n\n"
            for _, r in wc_og.sort_values("date",ascending=False).head(10).iterrows():
                out += (f"  - {r['scorer']} ({r['team']}) — "
                        f"{r['home_team']} vs {r['away_team']} [{str(r['date'])[:10]}]\n")
            return out

        if any(x in q for x in ["most goals in a tournament","goals per tournament","highest scoring tournament"]):
            tg = (wc_matches.groupby("tournament Name")
                            .apply(lambda x: int((x["Home Team Score"]+x["Away Team Score"]).sum()))
                            .reset_index(name="goals").sort_values("goals",ascending=False))
            out = "World Cup tournaments by total goals:\n\n"
            for i,(_, r) in enumerate(tg.iterrows(),1):
                m = len(wc_matches[wc_matches["tournament Name"]==r["tournament Name"]])
                out += (f"  {i:>2}. {r['tournament Name']}: "
                        f"{int(r['goals'])} goals ({m} matches, "
                        f"{round(r['goals']/m,2)} per game)\n")
            return out

        return (
            f"Analytics summary:\n"
            f"  Total WC matches  : {len(wc_matches)}\n"
            f"  Total goals       : {int(wc_matches['Home Team Score'].sum()+wc_matches['Away Team Score'].sum())}\n"
            f"  Tournaments       : {wc_matches['tournament Name'].nunique()}\n"
            f"  Shootouts         : {len(shootouts)}\n"
            f"  Goalscorer records: {len(goalscorers)}\n\n"
            f"Try: hat-tricks, highest scoring matches, biggest wins, "
            f"most appearances, shootout records, own goals, tournament statistics."
        )

    tools_ = [
        dataset_discovery_tool,
        data_ingestion_tool,
        retrieval_or_filter_tool,
        reasoning_or_aggregation_tool,
        llm_synthesis_tool,
        report_generation_tool,
        analytics_tool,
    ]

    mem = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    mem.chat_memory.add_user_message("Keep all predictions strictly data-driven and unbiased.")
    mem.chat_memory.add_ai_message("Understood. All responses grounded in historical data only.")
    mem.chat_memory.add_user_message("Always state data limitations at the end of predictions.")
    mem.chat_memory.add_ai_message("Noted. Limitations disclaimer included in every prediction.")

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are an expert FIFA World Cup analyst, football historian, and
match prediction assistant with deep knowledge of global football.

CRITICAL ANTI-HALLUCINATION RULES — follow these exactly:
1. NEVER state a statistic you have not retrieved from a tool in this conversation.
2. ALWAYS call reasoning_or_aggregation_tool FIRST for ANY prediction or comparison.
3. ALWAYS call data_ingestion_tool before stating any team's win rate, goals, or record.
4. NEVER invent ELO scores, FIFA ratings, win percentages, or match results.
5. If a tool returns no data, say so explicitly — do not substitute with guesses.
6. When giving a prediction, clearly separate: (a) what the data shows, (b) your conclusion.
7. Win rates and form stats from tools are HISTORICAL — label them as such.
8. Always end predictions with: data covers up to 2022 only.

You also have broad football knowledge — use it to enrich answers with context,
narratives, and tactical insights, but clearly distinguish it from dataset facts.
Say "based on my knowledge" vs "based on dataset" when mixing sources.

TEMPORAL CONTEXT:
- Most recent completed World Cup: Qatar 2022 — Argentina beat France on penalties.
- Next World Cup: USA, Canada and Mexico 2026 — not yet played.
- For ANY "Predict X vs Y" request, ALWAYS call reasoning_or_aggregation_tool
  regardless of whether the match is historical or hypothetical.
  Never refuse a prediction — always use the data tools.

Tool usage:
- dataset_discovery_tool        : what data or teams are available
- data_ingestion_tool           : REQUIRED before stating any team stat
- retrieval_or_filter_tool      : historical matches, tournaments, events — primary fallback
- reasoning_or_aggregation_tool : REQUIRED for all predictions and comparisons
- llm_synthesis_tool            : top scorers, player-level data
- report_generation_tool        : full structured match preview report
- analytics_tool                : hat-tricks, biggest wins, tournament stats, raw computations

STYLE RULES:
- Be dynamic, engaging and conversational — not robotic or overly cautious.
- Do not use markdown headers. Use bold text and bullet points only.
- Always note when predictions are data-driven estimates, not guarantees.
{style_instruction}
Educational purposes only. Not professional sports analytics advice."""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    # ── FIXED: temperature=0.2, max_tokens=1500, max_iterations=8 ──
    llm_   = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2, max_tokens=1500)
    agent  = create_tool_calling_agent(llm_, tools_, prompt)
    return AgentExecutor(agent=agent, tools=tools_, memory=mem,
                         verbose=False, max_iterations=8, handle_parsing_errors=True)

# ── Agent cache ──────────────────────────────────────────────
_agent_cache = {}
def get_agent(style):
    if style not in _agent_cache:
        _agent_cache[style] = make_agent(style)
    return _agent_cache[style]

# ── Team stats helper ────────────────────────────────────────
def get_stats_for_team(team: str):
    t     = resolve_team(team)
    h     = wc_matches[wc_matches["Home Team Name"] == t]
    a     = wc_matches[wc_matches["Away Team Name"] == t]
    total = len(h) + len(a)
    if total == 0:
        return None
    wins   = len(h[h["Home Team Win"]==1]) + len(a[a["Away Team Win"]==1])
    draws  = len(h[h["Draw"]==1])          + len(a[a["Draw"]==1])
    gf     = int(h["Home Team Score"].sum() + a["Away Team Score"].sum())
    ga     = int(h["Away Team Score"].sum() + a["Home Team Score"].sum())
    fm     = get_team_form(t)
    return {
        "team"              : t,
        "total"             : total,
        "wins"              : wins,
        "draws"             : draws,
        "losses"            : total - wins - draws,
        "goals_for"         : gf,
        "goals_against"     : ga,
        "goal_diff"         : gf - ga,
        "win_pct"           : round(wins / total * 100, 1),
        "recent_win_rate"   : fm.get("win_rate", "N/A"),
        "avg_goals_scored"  : fm.get("avg_goals_scored", "N/A"),
        "avg_goals_conceded": fm.get("avg_goals_conceded", "N/A"),
        "shootout_wins"     : len(shootouts[shootouts["winner"] == t]),
    }

# ── API Endpoints ────────────────────────────────────────────
@app.get("/")
def root():
    return {"status": "Personal FIFA World Cup Analyst API", "tools": 7}

@app.get("/health")
def health():
    return {"status": "ok", "vectors": vectorstore.index.ntotal, "tools": 7}

@app.post("/chat")
async def chat(req: ChatRequest):
    try:
        agent  = get_agent(req.style)
        result = agent.invoke({"input": req.message})
        return {"response": result["output"], "status": "ok"}
    except Exception as e:
        return {"response": f"Error: {str(e)}", "status": "error"}

@app.get("/team-stats/{team}")
def team_stats(team: str):
    resolved = resolve_team(team)
    stats    = get_stats_for_team(resolved)
    if not stats:
        all_teams = sorted(set(
            list(wc_matches["Home Team Name"].unique()) +
            list(wc_matches["Away Team Name"].unique())
        ))
        matches = [t for t in all_teams if team.lower() in t.lower()]
        if matches:
            stats = get_stats_for_team(matches[0])
        if not stats:
            return {"status": "not_found", "team": team,
                    "hint": f"Try: {', '.join(all_teams[:10])}..."}
    return {"status": "ok", "data": stats}

@app.get("/teams")
def list_teams():
    teams = sorted(set(
        list(wc_matches["Home Team Name"].unique()) +
        list(wc_matches["Away Team Name"].unique())
    ))
    return {"teams": teams}

@app.get("/chart/h2h/{team_a}/{team_b}")
def chart_h2h(team_a: str, team_b: str):
    ta, tb = resolve_team(team_a), resolve_team(team_b)
    mask = (
        ((results["home_team"]==ta)&(results["away_team"]==tb)) |
        ((results["home_team"]==tb)&(results["away_team"]==ta))
    )
    h2h = results[mask]
    if h2h.empty:
        return {"status": "not_found"}
    a_wins = len(h2h[((h2h["home_team"]==ta)&(h2h["result"]=="home"))|((h2h["away_team"]==ta)&(h2h["result"]=="away"))])
    b_wins = len(h2h[((h2h["home_team"]==tb)&(h2h["result"]=="home"))|((h2h["away_team"]==tb)&(h2h["result"]=="away"))])
    draws  = len(h2h[h2h["result"]=="draw"])
    return {
        "status":"ok","type":"bar","title":f"{ta} vs {tb} - Head to Head",
        "labels":[f"{ta} Wins","Draws",f"{tb} Wins"],
        "values":[a_wins,draws,b_wins],
        "colors":["#2d6a4f","#c9a84c","#1a3a5c"]
    }

@app.get("/chart/team-form/{team}")
def chart_team_form(team: str):
    t  = resolve_team(team)
    tf = form[form["team"]==t].sort_values("match_date").tail(20)
    if tf.empty:
        return {"status":"not_found"}
    return {
        "status":"ok","type":"line","title":f"{t} - Recent Form",
        "labels":[str(d)[:10] for d in tf["match_date"]],
        "datasets":[
            {"label":"Win Rate %","values":[round(float(v)*100,1) for v in tf["win_rate"]],"color":"#2d6a4f"},
            {"label":"Avg Scored","values":[round(float(v),2) for v in tf["avg_goals_scored"]],"color":"#c9a84c"},
            {"label":"Avg Conceded","values":[round(float(v),2) for v in tf["avg_goals_conceded"]],"color":"#e74c3c"},
        ]
    }

@app.get("/chart/top-scorers/{team}")
def chart_top_scorers(team: str):
    t = resolve_team(team) if team.lower() != "all" else "all"
    wc_gs = WC_GOALS_ONLY.copy()
    if t != "all":
        wc_gs = wc_gs[wc_gs["team"] == t]
    top = (wc_gs.groupby("scorer").size().reset_index(name="goals")
               .sort_values("goals",ascending=False).head(10))
    if top.empty:
        return {"status":"not_found"}
    title = f"Top WC Scorers - {t}" if t != "all" else "All-Time Top WC Scorers"
    return {
        "status":"ok","type":"bar_h","title":title,
        "labels":list(top["scorer"])[::-1],
        "values":list(top["goals"].astype(int))[::-1],
        "colors":["#c9a84c"]*len(top)
    }

@app.get("/chart/tournament-goals")
def chart_tournament_goals():
    tg = (wc_matches.groupby("tournament Name")
                    .apply(lambda x: int((x["Home Team Score"]+x["Away Team Score"]).sum()))
                    .reset_index(name="goals").sort_values("goals",ascending=False))
    return {
        "status":"ok","type":"bar","title":"Total Goals by World Cup Tournament",
        "labels":list(tg["tournament Name"]),
        "values":list(tg["goals"]),
        "colors":["#2d6a4f"]*len(tg)
    }

@app.get("/chart/team-radar/{team_a}/{team_b}")
def chart_radar(team_a: str, team_b: str):
    ta, tb = resolve_team(team_a), resolve_team(team_b)
    def get_feat(t):
        sub = features[(features["_home_team"]==t)|(features["_away_team"]==t)]
        if sub.empty:
            return None
        row    = sub.sort_values("_date",ascending=False).iloc[0]
        prefix = "home" if row["_home_team"]==t else "away"
        return {k: float(row.get(f"{prefix}_avg_{k}",0))
                for k in ["overall","attack","defense","pace","shooting","passing"]}
    fa, fb = get_feat(ta), get_feat(tb)
    if not fa or not fb:
        return {"status":"not_found"}
    cats = ["Overall","Attack","Defense","Pace","Shooting","Passing"]
    return {
        "status":"ok","type":"radar","title":f"{ta} vs {tb} - FIFA Ratings",
        "categories":cats,
        "datasets":[
            {"label":ta,"values":[fa["overall"],fa["attack"],fa["defense"],fa["pace"],fa["shooting"],fa["passing"]],"color":"#2d6a4f"},
            {"label":tb,"values":[fb["overall"],fb["attack"],fb["defense"],fb["pace"],fb["shooting"],fb["passing"]],"color":"#c9a84c"},
        ]
    }

# ── Launch ───────────────────────────────────────────────────
print("🚀 Starting FastAPI server...")

def run():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error", loop="asyncio")

threading.Thread(target=run, daemon=True).start()
time.sleep(3)

print("✅ FastAPI running on port 8000")
print("📡 API docs: http://localhost:8000/docs")
print("\nPipeline tools registered:")
for name in ["dataset_discovery_tool","data_ingestion_tool","retrieval_or_filter_tool",
             "reasoning_or_aggregation_tool","llm_synthesis_tool","report_generation_tool",
             "analytics_tool"]:
    print(f"  ✅ {name}")
print("\n🚀 Block 6A Complete — Run Block 6B to open ngrok tunnel")

✅ Port 8000 ready.
🚀 Starting FastAPI server...


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use


✅ FastAPI running on port 8000
📡 API docs: http://localhost:8000/docs

Pipeline tools registered:
  ✅ dataset_discovery_tool
  ✅ data_ingestion_tool
  ✅ retrieval_or_filter_tool
  ✅ reasoning_or_aggregation_tool
  ✅ llm_synthesis_tool
  ✅ report_generation_tool
  ✅ analytics_tool

🚀 Block 6A Complete — Run Block 6B to open ngrok tunnel


In [12]:
# ============================================================
# BLOCK 6B: Expose FastAPI via ngrok
# Kills all existing tunnels first to avoid the 5-limit error
# ============================================================
from pyngrok import ngrok
import requests, time

# Kill all existing tunnels AND the ngrok service to ensure a clean start
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
    print(f"  🔌 Closed: {tunnel.public_url}")

ngrok.kill()
time.sleep(1)

# Open a single fresh tunnel
public_url = ngrok.connect(8000)
api_url    = public_url.public_url.strip()

# Health check
time.sleep(2)
try:
    r = requests.get(f"{api_url}/health",
                     headers={"ngrok-skip-browser-warning": "true"},
                     timeout=10)
    health = r.json()
    print(f"\n{'='*58}")
    print(f"  ⚽  Personal FIFA World Cup Analyst — API LIVE")
    print(f"{'='*58}")
    print(f"  🌐  Public API URL : {api_url}")
    print(f"  📊  Vectors loaded : {health.get('vectors', '?')}")
    print(f"  🔧  Tools active   : {health.get('tools', '?')}")
    print(f"  ✅  Status         : {health.get('status', '?')}")
    print(f"{'='*58}")
    print(f"\n  📋  Paste this into Vercel ⚙ Set API URL:")
    print(f"  👉  {api_url}")
    print(f"\n  ℹ️   Active while this Colab session runs.")
except Exception as e:
    print(f"❌ Health check failed: {e}")
    print(f"   API URL: {api_url}")

  🔌 Closed: https://unhelpful-jami-unflamboyantly.ngrok-free.dev

  ⚽  Personal FIFA World Cup Analyst — API LIVE
  🌐  Public API URL : https://unhelpful-jami-unflamboyantly.ngrok-free.dev
  📊  Vectors loaded : 1047
  🔧  Tools active   : 7
  ✅  Status         : ok

  📋  Paste this into Vercel ⚙ Set API URL:
  👉  https://unhelpful-jami-unflamboyantly.ngrok-free.dev

  ℹ️   Active while this Colab session runs.
